# Multi-database LCIA of European hydrogen markets

This notebook calculates selected **EF 3.1 midpoint** and **premise hydrogen-inclusive GWP100** results for the generic European low-pressure hydrogen market and every available European sector-specific market. It then performs a reconciled production/distribution contribution analysis and compares years and IAM scenarios with one consistent visual language.

The functional unit is **1 kg hydrogen, gaseous, low pressure**. Every raw result, selected dataset, classification decision, grouped contribution, and reconciliation check is exported to CSV before plotting.


## Workflow

1. Edit only the configuration and mapping cells at the beginning.
2. Validate the Brightway project, database records, exact LCIA methods, and scenario metadata.
3. Calculate raw LCIA scores for every selected database and market.
4. Classify direct hydrogen-market branches into production and distribution groups.
5. Export raw scores and all audit/reconciliation tables.
6. Plot signed production/distribution shares over time.
7. Compare IAM scenarios and years relative to a configured reference year.

Missing sector markets are retained as an auditable scenario outcome; `premise` creates them only where the modeled sector has positive hydrogen demand and the required production activities exist.


In [ ]:
import os

os.environ["MKL_THREADING_LAYER"] = "TBB"

from pathlib import Path
import math
import sys
import warnings

_CONDA_DLL_DIR = Path(sys.executable).resolve().parent / "Library" / "bin"
_CONDA_DLL_HANDLE = None
if os.name == "nt" and _CONDA_DLL_DIR.exists():
    _CONDA_DLL_HANDLE = os.add_dll_directory(str(_CONDA_DLL_DIR))
    _mkl_runtime = _CONDA_DLL_DIR / "mkl_rt.2.dll"
    if _mkl_runtime.exists():
        os.environ["PYPARDISO_MKL_RT"] = str(_mkl_runtime)

import bw2calc as bc
import bw2data as bd
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from premise_gwp import add_premise_gwp
from IPython.display import display

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "examples" / "h2-distribution_LCIA",
    Path.cwd().parent,
]
ANALYSIS_DIR = next(
    (path.resolve() for path in candidate_dirs if (path / "config.py").exists()),
    None,
)
if ANALYSIS_DIR is None:
    raise FileNotFoundError(
        "Run this notebook from the repository root, the h2-distribution_LCIA "
        "folder, or its notebooks folder."
    )
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

import config as cfg
from mapping import impact_category_label, normalized
from run_analysis import (
    analyze_hydrogen_life_cycle_stages,
    calculate_lcia,
    select_markets,
)

pd.set_option("display.max_colwidth", 140)


## 1. Editable analysis configuration

This is the main user-editable block. Database metadata are explicit rather than parsed from filenames. Add one record per Brightway database; records sharing the same IAM model and scenario are treated as a time series.

Impact methods are exact Brightway tuples. Add or remove entries from `METHOD_LABELS` to change the raw LCIA calculation. Contribution analysis can be run for any selected method, but premise GWP is the default because it characterizes hydrogen emissions to air.


In [ ]:
PROJECT = "ecoinvent-3.12-cutoff"

DATABASES = [
    {
        "database": "ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2030_06-2_full",
        "iam_model": "remind",
        "scenario": "SSP1-PkBudg650",
        "year": 2030,
    },
    {
        "database": "ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2040_06-2_full",
        "iam_model": "remind",
        "scenario": "SSP1-PkBudg650",
        "year": 2040,
    },
    {
        "database": "ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2050_06-2_full",
        "iam_model": "remind",
        "scenario": "SSP1-PkBudg650",
        "year": 2050,
    },
]

# Scenario metadata follow the explicit mapping style used in Figure_7_CA.ipynb.
MODEL_LABELS = {
    "image": "IMAGE",
    "message": "MESSAGE",
    "remind": "REMIND",
}
SCENARIO_LABELS = {
    ("remind", "SSP1-PkBudg650"): "REMIND — SSP1-PkBudg650",
}
WARMING_MAP = {
    ("remind", "SSP1-PkBudg650"): "<2.0 °C",
}

PREMISE_GWP_METHOD = ("IPCC 2021", "climate change", "GWP 100a, incl. H")
METHOD_LABELS = {
    ("EF v3.1", "acidification", "accumulated exceedance (AE)"): "Acidification",
    ("EF v3.1", "climate change", "global warming potential (GWP100)"): "Climate change — EF 3.1",
    ("EF v3.1", "ecotoxicity: freshwater", "comparative toxic unit for ecosystems (CTUe)"): "Ecotoxicity — freshwater",
    ("EF v3.1", "energy resources: non-renewable", "abiotic depletion potential (ADP): fossil fuels"): "Resource use — energy carriers",
    ("EF v3.1", "eutrophication: freshwater", "fraction of nutrients reaching freshwater end compartment (P)"): "Eutrophication — freshwater",
    ("EF v3.1", "eutrophication: marine", "fraction of nutrients reaching marine end compartment (N)"): "Eutrophication — marine",
    ("EF v3.1", "eutrophication: terrestrial", "accumulated exceedance (AE)"): "Eutrophication — terrestrial",
    ("EF v3.1", "human toxicity: carcinogenic", "comparative toxic unit for human (CTUh)"): "Human toxicity — cancer",
    ("EF v3.1", "human toxicity: non-carcinogenic", "comparative toxic unit for human (CTUh)"): "Human toxicity — non-cancer",
    ("EF v3.1", "ionising radiation: human health", "human exposure efficiency relative to u235"): "Ionising radiation — human health",
    ("EF v3.1", "land use", "soil quality index"): "Land use",
    ("EF v3.1", "material resources: metals/minerals", "abiotic depletion potential (ADP): elements (ultimate reserves)"): "Resource use — minerals and metals",
    ("EF v3.1", "ozone depletion", "ozone depletion potential (ODP)"): "Ozone depletion",
    ("EF v3.1", "particulate matter formation", "impact on human health"): "Particulate matter",
    ("EF v3.1", "photochemical oxidant formation: human health", "tropospheric ozone concentration increase"): "Photochemical ozone formation",
    ("EF v3.1", "water use", "user deprivation potential (deprivation-weighted water consumption)"): "Water use",
    PREMISE_GWP_METHOD: "Climate change — premise GWP incl. H₂",
}
SELECTED_METHODS = list(METHOD_LABELS)
CONTRIBUTION_METHODS = [PREMISE_GWP_METHOD]

OUTPUT_DIR = ANALYSIS_DIR / "results" / "multi_iam_hydrogen_markets"
EXPORT_RESULTS = True
SAVE_PLOTS = True
FIGURE_DPI = 300
USE_SCIPY_SOLVER = False  # set True to bypass pypardiso and use SciPy SuperLU

# Keep a complete market grid. Missing scenario markets are exported and plotted as NA.
ANALYSIS_MARKETS = [
    "Generic", "Transport", "Chemicals", "Steel",
    "Cement", "Heating", "Other end uses",
]
CONTRIBUTION_MARKETS = ANALYSIS_MARKETS
SCENARIO_COMPARISON_MARKETS = ["Generic", "Steel"]
COMPARISON_REFERENCE_YEAR = 2030


## 2. Editable process mapping and visual specification

The detailed production/distribution classification is inherited from `mapping.py`, where exact Brightway activity names are mapped and unknown direct market inputs fail loudly. The rules below form the second, plot-oriented layer: they group detailed distribution processes into transport technologies.

Each transport technology has one hatch and one color family. Individual processes within that technology receive deterministic shades from the same family. Unknown processes raise an error listing the unmapped names; update `DISTRIBUTION_FAMILY_RULES` rather than allowing a silent fallback.


In [ ]:
DISTRIBUTION_FAMILY_RULES = {
    "Compressed gas truck": (
        "gaseous h2 truck",
        "compression (lorry)",
    ),
    "Pipeline": (
        "pipeline distribution",
        "compression (pipeline)",
    ),
    "Liquid hydrogen": (
        "liquid h2 truck",
        "liquid h2 tanker",
        "liquefaction",
        "regasification",
    ),
    "Ammonia shipping": (
        "ammonia tanker",
        "ammonia production",
        "ammonia cracking",
        "ammonia leakage",
    ),
    "Shared distribution": (
        "hydrogen leakage — distribution",
        "hydrogen leakage - distribution",
    ),
}

FAMILY_STYLES = {
    "Hydrogen production": {"cmap": "Greys", "hatch": ""},
    "Compressed gas truck": {"cmap": "Blues", "hatch": "///"},
    "Pipeline": {"cmap": "Greens", "hatch": "\\\\"},
    "Liquid hydrogen": {"cmap": "Oranges", "hatch": "xx"},
    "Ammonia shipping": {"cmap": "Purples", "hatch": "oo"},
    "Shared distribution": {"cmap": "Reds", "hatch": ".."},
}
FAMILY_ORDER = list(FAMILY_STYLES)

YEAR_COLORS = {
    2030: "#c8d9ef",
    2040: "#6baed6",
    2050: "#ec006d",
}
WARMING_COLORS = {
    "<2.0 °C": "#1B5E20",
    "2.0-2.5 °C": "#FF9800",
    ">2.5 °C": "#8B0000",
}

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.titleweight": "bold",
    "axes.labelsize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "legend.title_fontsize": 14,
    "figure.titlesize": 18,
    "hatch.linewidth": 0.7,
})


## 3. Validate configuration and Brightway context

The project is checked before activation because `set_current` would otherwise create a misspelled project. Methods, database names, scenario mappings, and year-series uniqueness are all validated before any LCIA calculation begins.


In [ ]:
REQUIRED_DATABASE_FIELDS = {"database", "iam_model", "scenario", "year"}
if not DATABASES:
    raise ValueError("DATABASES must contain at least one database record.")
for index, record in enumerate(DATABASES):
    missing = REQUIRED_DATABASE_FIELDS - set(record)
    if missing:
        raise KeyError(f"DATABASES[{index}] is missing fields: {sorted(missing)}")

database_config_df = pd.DataFrame(DATABASES).copy()
database_config_df["iam_model"] = database_config_df["iam_model"].str.lower()
database_config_df["year"] = database_config_df["year"].astype(int)
if database_config_df["database"].duplicated().any():
    duplicates = database_config_df.loc[
        database_config_df["database"].duplicated(False), "database"
    ].tolist()
    raise ValueError(f"Duplicate Brightway database names: {duplicates}")
series_key = ["iam_model", "scenario", "year"]
if database_config_df.duplicated(series_key).any():
    raise ValueError(
        "Each IAM model/scenario/year combination must identify exactly one database."
    )

scenario_keys = list(zip(database_config_df["iam_model"], database_config_df["scenario"]))
missing_scenario_labels = sorted(set(scenario_keys) - set(SCENARIO_LABELS))
missing_warming_bins = sorted(set(scenario_keys) - set(WARMING_MAP))
missing_model_labels = sorted(set(database_config_df["iam_model"]) - set(MODEL_LABELS))
if missing_scenario_labels:
    raise KeyError(f"Add SCENARIO_LABELS entries for: {missing_scenario_labels}")
if missing_warming_bins:
    raise KeyError(f"Add WARMING_MAP entries for: {missing_warming_bins}")
if missing_model_labels:
    raise KeyError(f"Add MODEL_LABELS entries for: {missing_model_labels}")

database_config_df["model label"] = database_config_df["iam_model"].map(MODEL_LABELS)
database_config_df["scenario label"] = [SCENARIO_LABELS[key] for key in scenario_keys]
database_config_df["warming bin"] = [WARMING_MAP[key] for key in scenario_keys]
database_config_df["scenario key"] = (
    database_config_df["iam_model"] + " | " + database_config_df["scenario"]
)

project_names = {project.name for project in bd.projects}
if PROJECT not in project_names:
    raise KeyError(f"Brightway project {PROJECT!r} does not exist. Available: {sorted(project_names)}")
bd.projects.set_current(PROJECT)
add_premise_gwp()

missing_databases = sorted(set(database_config_df["database"]) - set(bd.databases))
if missing_databases:
    raise KeyError(f"Configured Brightway databases are unavailable: {missing_databases}")
missing_methods = [method for method in SELECTED_METHODS if method not in bd.methods]
if missing_methods:
    raise LookupError(f"Configured LCIA methods are unavailable: {missing_methods}")
if not set(CONTRIBUTION_METHODS).issubset(SELECTED_METHODS):
    raise ValueError("Every CONTRIBUTION_METHOD must also be present in SELECTED_METHODS.")

if USE_SCIPY_SOLVER:
    import bw2calc.lca_base as lca_base
    from scipy.sparse.linalg import factorized as scipy_factorized
    from scipy.sparse.linalg import spsolve as scipy_spsolve

    bc.PYPARDISO = False
    bc.spsolve = scipy_spsolve
    bc.factorized = scipy_factorized
    lca_base.PYPARDISO = False
    lca_base.spsolve = scipy_spsolve
    lca_base.factorized = scipy_factorized
    print("Sparse solver: scipy.sparse.linalg")

method_units = {
    method: bd.Method(method).metadata.get("unit", "")
    for method in SELECTED_METHODS
}
method_config_df = pd.DataFrame([
    {
        "method": method,
        "impact label": METHOD_LABELS[method],
        "unit": method_units[method],
        "run contribution analysis": method in CONTRIBUTION_METHODS,
    }
    for method in SELECTED_METHODS
])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project: {bd.projects.current}")
print(f"Configured databases: {len(database_config_df)}")
print(f"Selected LCIA methods: {len(SELECTED_METHODS)}")
display(database_config_df)
display(method_config_df)


## 4. Raw LCIA and reconciled contribution calculations

For each database, market selection requires exact activity name, reference product, unit, and location. The generic market must exist exactly once at `RER`; sector markets may use the configured `EUR`/`WEU` fallback. The raw LCI is solved once per market and the characterization method is switched for the selected LCIA categories.

Contribution analysis preserves signed credits and leakage burdens. The existing stage engine verifies that Layer 1 and Layer 2 contributions reconstruct each market score within `RECONCILIATION_RTOL`.


In [ ]:
def attach_database_metadata(frame, record):
    result = frame.copy()
    metadata = {
        "database": record["database"],
        "iam model": record["iam_model"],
        "model label": record["model label"],
        "scenario": record["scenario"],
        "scenario label": record["scenario label"],
        "scenario key": record["scenario key"],
        "warming bin": record["warming bin"],
        "year": int(record["year"]),
    }
    for column, value in reversed(list(metadata.items())):
        result.insert(0, column, value)
    return result


def complete_selection_table(selection):
    complete = pd.DataFrame({"comparison label": ANALYSIS_MARKETS}).merge(
        selection, on="comparison label", how="left", validate="one_to_one"
    )
    complete["market availability"] = np.where(
        complete["database key"].notna(),
        "available",
        "NA — market not generated",
    )
    return complete


def complete_raw_score_table(scores):
    grid = pd.MultiIndex.from_product(
        [ANALYSIS_MARKETS, SELECTED_METHODS], names=["market", "method"]
    ).to_frame(index=False)
    complete = grid.merge(
        scores, on=["market", "method"], how="left", validate="one_to_one"
    )
    complete["impact category"] = complete["method"].map(impact_category_label)
    complete["impact label"] = complete["method"].map(METHOD_LABELS)
    complete["indicator"] = complete["method"].map(lambda method: method[-1])
    complete["unit"] = complete["method"].map(method_units)
    complete["market availability"] = np.where(
        complete["score per kg H2"].notna(),
        "available",
        "NA — market not generated",
    )
    return complete


def distribution_family(process):
    text = normalized(process)
    matches = [
        family
        for family, patterns in DISTRIBUTION_FAMILY_RULES.items()
        if any(normalized(pattern) in text for pattern in patterns)
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one transport-technology mapping for {process!r}; "
            f"matched {matches}. Update DISTRIBUTION_FAMILY_RULES."
        )
    return matches[0]


def combined_component_table(stage, scores, method):
    production = stage["layer1"].loc[
        stage["layer1"]["layer 1 group"].eq("Production technology")
    ].copy()
    production["process"] = production["component"]
    production["stage"] = "Production"
    production["transport technology"] = "Hydrogen production"

    distribution = stage["distribution processes"].copy()
    distribution["component"] = distribution["process"]
    distribution["stage"] = "Distribution"
    distribution["transport technology"] = distribution["process"].map(
        distribution_family
    )

    columns = [
        "market", "stage", "transport technology", "component", "process",
        "contribution type", "score", "share of market (%)", "method",
        "impact category", "unit",
    ]
    combined = pd.concat(
        [production.reindex(columns=columns), distribution.reindex(columns=columns)],
        ignore_index=True,
    )

    expected = (
        scores.loc[scores["method"].isin([method])]
        .set_index("market")["score per kg H2"]
        .sort_index()
    )
    reconstructed = combined.groupby("market")["score"].sum().reindex(expected.index)
    if not np.allclose(
        reconstructed.to_numpy(dtype=float),
        expected.to_numpy(dtype=float),
        rtol=cfg.RECONCILIATION_RTOL,
        atol=1e-12,
    ):
        check = pd.DataFrame({"expected": expected, "reconstructed": reconstructed})
        check["difference"] = check["reconstructed"] - check["expected"]
        raise AssertionError(
            "Production technologies plus detailed distribution processes do not "
            f"reconstruct the market scores:\n{check}"
        )
    return combined


frames = {
    "selection": [],
    "raw scores": [],
    "top process contributions": [],
    "component contributions": [],
    "classification audit": [],
    "stage reconciliation": [],
}
market_order_by_database = {}

for record in database_config_df.to_dict("records"):
    database_name = record["database"]
    print(f"Running {record['scenario label']} — {record['year']}\n  {database_name}")
    database = bd.Database(database_name)
    selected, market_order, sector_order, selection = select_markets(database)
    market_order_by_database[database_name] = market_order

    scores, top_processes = calculate_lcia(
        selected=selected,
        market_order=market_order,
        lcia_methods=SELECTED_METHODS,
        method_units=method_units,
    )
    scores["impact label"] = scores["method"].map(METHOD_LABELS)
    top_processes["impact label"] = top_processes["method"].map(METHOD_LABELS)
    complete_selection = complete_selection_table(selection)
    complete_scores = complete_raw_score_table(scores)
    frames["selection"].append(attach_database_metadata(complete_selection, record))
    frames["raw scores"].append(attach_database_metadata(complete_scores, record))
    frames["top process contributions"].append(
        attach_database_metadata(top_processes, record)
    )

    for method in CONTRIBUTION_METHODS:
        stage = analyze_hydrogen_life_cycle_stages(
            selected=selected,
            market_order=market_order,
            method=method,
            scores_df=scores,
            impact_category=impact_category_label(method),
            unit=method_units[method],
            reconciliation_rtol=cfg.RECONCILIATION_RTOL,
        )
        components = combined_component_table(stage, scores, method)
        components["impact label"] = METHOD_LABELS[method]
        frames["component contributions"].append(
            attach_database_metadata(components, record)
        )
        frames["classification audit"].append(
            attach_database_metadata(stage["classification audit"], record)
        )
        frames["stage reconciliation"].append(
            attach_database_metadata(stage["reconciliation"], record)
        )
    print("  finished")

results = {
    name: pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    for name, parts in frames.items()
}
raw_scores_df = results["raw scores"]
component_contributions_df = results["component contributions"]


In [ ]:
expected_raw_rows = (
    len(database_config_df) * len(ANALYSIS_MARKETS) * len(SELECTED_METHODS)
)
if len(raw_scores_df) != expected_raw_rows:
    raise AssertionError(
        f"Expected {expected_raw_rows} raw LCIA rows; found {len(raw_scores_df)}."
    )
if raw_scores_df["impact label"].isna().any():
    raise AssertionError("One or more raw LCIA methods have no display label.")

exported_paths = {}
export_tables = {
    "database configuration": database_config_df,
    "method configuration": method_config_df,
    **results,
}
if EXPORT_RESULTS:
    filenames = {
        "database configuration": "00_database_configuration.csv",
        "method configuration": "01_method_configuration.csv",
        "selection": "02_market_selection_audit.csv",
        "raw scores": "03_raw_lcia_scores_per_kg_h2.csv",
        "top process contributions": "04_top_process_contributions.csv",
        "component contributions": "05_grouped_component_contributions.csv",
        "classification audit": "06_stage_classification_audit.csv",
        "stage reconciliation": "07_stage_reconciliation.csv",
    }
    for label, table in export_tables.items():
        path = OUTPUT_DIR / filenames[label]
        table.to_csv(path, index=False)
        exported_paths[label] = path
    if len(set(exported_paths.values())) != len(exported_paths):
        raise AssertionError("Two exports resolve to the same path.")

print(f"Raw LCIA rows: {len(raw_scores_df):,}")
print(f"Grouped contribution rows: {len(component_contributions_df):,}")
display(results["selection"])
display(results["stage reconciliation"])
display(raw_scores_df.head())


## 5. Global component styles

Styles are generated once from the union of all calculated databases. Sorting component names makes the assigned shades stable when notebook execution order changes. A component therefore keeps the same color and hatch in every market, year, scenario, and figure.


In [ ]:
def component_style_registry(data):
    registry = {}
    unknown_families = sorted(set(data["transport technology"]) - set(FAMILY_STYLES))
    if unknown_families:
        raise KeyError(f"Add FAMILY_STYLES entries for: {unknown_families}")

    for family in FAMILY_ORDER:
        components = sorted(
            data.loc[data["transport technology"].eq(family), "component"]
            .dropna()
            .astype(str)
            .unique()
        )
        if not components:
            continue
        cmap = plt.get_cmap(FAMILY_STYLES[family]["cmap"])
        shades = np.linspace(0.42, 0.82, len(components))
        for component, shade in zip(components, shades):
            registry[component] = {
                "family": family,
                "color": mcolors.to_hex(cmap(shade)),
                "hatch": FAMILY_STYLES[family]["hatch"],
            }
    return registry


COMPONENT_STYLE = component_style_registry(component_contributions_df)
COMPONENT_ORDER = [
    component
    for family in FAMILY_ORDER
    for component in sorted(
        component
        for component, style in COMPONENT_STYLE.items()
        if style["family"] == family
    )
]
assert set(COMPONENT_ORDER) == set(COMPONENT_STYLE)


def safe_filename(value):
    cleaned = "".join(character if character.isalnum() else "_" for character in str(value))
    return "_".join(part for part in cleaned.split("_") if part).lower()


def finish_figure(fig, filename, *, rect=None):
    if rect is None:
        fig.tight_layout()
    else:
        fig.tight_layout(rect=rect)
    if SAVE_PLOTS:
        path = OUTPUT_DIR / filename
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
        exported_paths[filename] = path
        print(f"Saved {path}")
    plt.show()


style_rows = [
    {"component": component, **style}
    for component, style in COMPONENT_STYLE.items()
]
display(pd.DataFrame(style_rows).sort_values(["family", "component"]))


## 6. Production and distribution shares over time

Rows are hydrogen markets and columns are IAM model/scenario combinations. Every year is shown as a signed stacked bar in the same subplot. Production technologies use grey shades; transport, conversion, reconversion, and leakage processes use technology-specific color families and hatches. The black diamond marks the reconstructed net share (normally 100%).


In [ ]:
def plot_contribution_shares_over_time(
    data,
    method=PREMISE_GWP_METHOD,
    markets=CONTRIBUTION_MARKETS,
):
    subset = data.loc[data["method"].isin([method])].copy()
    if subset.empty:
        raise ValueError(f"No contribution results are available for {method}.")

    configured_scenarios = database_config_df[
        ["scenario key", "scenario label", "warming bin"]
    ].drop_duplicates()
    scenario_order = configured_scenarios["scenario key"].tolist()
    scenario_title = configured_scenarios.set_index("scenario key")["scenario label"].to_dict()
    warming_by_scenario = configured_scenarios.set_index("scenario key")["warming bin"].to_dict()

    markets = list(markets)
    unknown_markets = sorted(set(markets) - set(ANALYSIS_MARKETS))
    if unknown_markets:
        raise KeyError(f"Unknown CONTRIBUTION_MARKETS entries: {unknown_markets}")

    fig = plt.figure(
        figsize=(max(15, 5.8 * len(scenario_order) + 7), max(12, 2.35 * len(markets)))
    )
    grid = fig.add_gridspec(
        len(markets),
        len(scenario_order) + 1,
        width_ratios=[1.0] * len(scenario_order) + [1.35],
        hspace=0.48,
        wspace=0.28,
    )
    axes = np.array([
        [fig.add_subplot(grid[row, column]) for column in range(len(scenario_order))]
        for row in range(len(markets))
    ])
    legend_ax = fig.add_subplot(grid[:, -1])
    legend_ax.axis("off")

    for row_index, market in enumerate(markets):
        row_axes = []
        for column_index, scenario_key in enumerate(scenario_order):
            ax = axes[row_index, column_index]
            row_axes.append(ax)
            panel = subset.loc[
                subset["market"].eq(market)
                & subset["scenario key"].eq(scenario_key)
            ]
            years = sorted(
                database_config_df.loc[
                    database_config_df["scenario key"].eq(scenario_key), "year"
                ].unique()
            )
            available_years = set(panel["year"].unique())

            x = np.arange(len(years), dtype=float)
            positive = np.zeros(len(years))
            negative = np.zeros(len(years))
            for component in COMPONENT_ORDER:
                values = (
                    panel.loc[panel["component"].eq(component)]
                    .groupby("year")["share of market (%)"]
                    .sum()
                    .reindex(years, fill_value=0.0)
                    .to_numpy(dtype=float)
                )
                if not np.any(values):
                    continue
                bottom = np.where(values >= 0, positive, negative)
                style = COMPONENT_STYLE[component]
                ax.bar(
                    x,
                    values,
                    width=0.65,
                    bottom=bottom,
                    color=style["color"],
                    hatch=style["hatch"],
                    edgecolor="white",
                    linewidth=0.5,
                    label=component,
                    zorder=3,
                )
                positive += np.where(values >= 0, values, 0.0)
                negative += np.where(values < 0, values, 0.0)

            net = (
                panel.groupby("year")["share of market (%)"]
                .sum()
                .reindex(years)
                .to_numpy(dtype=float)
            )
            ax.plot(
                x, net, marker="D", linestyle="None", color="black",
                markeredgecolor="white", markeredgewidth=0.7, markersize=6, zorder=5,
            )
            ax.axhline(0, color="black", linewidth=0.9, zorder=4)
            tick_labels = [
                str(year) if year in available_years else f"{year}\nNA"
                for year in years
            ]
            ax.set_xticks(x, tick_labels)
            ax.grid(axis="y", alpha=0.25, linestyle="--", zorder=0)
            ax.spines[["top", "right"]].set_visible(False)
            if row_index == 0:
                warming = warming_by_scenario[scenario_key]
                ax.set_title(
                    scenario_title[scenario_key],
                    color=WARMING_COLORS.get(warming, "#333333"),
                    pad=8,
                )
            if column_index == 0:
                ax.text(
                    -0.20, 0.5, market, transform=ax.transAxes,
                    fontsize=16, fontweight="bold", rotation=90,
                    ha="center", va="center",
                )
            if row_index == len(markets) - 1:
                ax.set_xlabel("Year")

        visible_axes = [ax for ax in row_axes if ax.axison]
        if visible_axes:
            ymin = min(ax.get_ylim()[0] for ax in visible_axes)
            ymax = max(ax.get_ylim()[1] for ax in visible_axes)
            padding = 0.04 * max(ymax - ymin, 1.0)
            for ax in visible_axes:
                ax.set_ylim(ymin - padding, ymax + padding)

    def legend_handles(components, include_family):
        return [
            mpatches.Patch(
                facecolor=COMPONENT_STYLE[component]["color"],
                hatch=COMPONENT_STYLE[component]["hatch"],
                edgecolor="#555555",
                linewidth=0.5,
                label=(
                    f"{COMPONENT_STYLE[component]['family']} — {component}"
                    if include_family else component
                ),
            )
            for component in components
        ]

    production_components = [
        component for component in COMPONENT_ORDER
        if COMPONENT_STYLE[component]["family"] == "Hydrogen production"
    ]
    distribution_components = [
        component for component in COMPONENT_ORDER
        if COMPONENT_STYLE[component]["family"] != "Hydrogen production"
    ]
    fig.supylabel("Share of market LCIA score (%)", x=0.025, fontsize=14)
    production_legend = legend_ax.legend(
        handles=legend_handles(production_components, include_family=False),
        title="Hydrogen production",
        loc="upper left",
        bbox_to_anchor=(0.0, 0.99),
        frameon=True, framealpha=0.95, fontsize=10,
    )
    legend_ax.add_artist(production_legend)
    legend_ax.legend(
        handles=legend_handles(distribution_components, include_family=True),
        title="Distribution processes",
        loc="upper left",
        bbox_to_anchor=(0.0, 0.56),
        frameon=True, framealpha=0.95, fontsize=10,
    )
    fig.suptitle(
        f"Production and distribution shares over time — {METHOD_LABELS[method]}",
        y=0.998,
    )
    finish_figure(
        fig,
        "01_production_distribution_shares_over_time.png",
        rect=(0.06, 0.03, 0.99, 0.97),
    )


plot_contribution_shares_over_time(component_contributions_df)


## 7. IAM scenario comparison in the Figure 7 layout

Selected impact categories form the shared y-axis; IAM model/scenario combinations form columns; years are grouped horizontal bars. Scores are normalized within each scenario and market as percentage change from `COMPARISON_REFERENCE_YEAR`:

\[
100 \times \frac{score_{year} - score_{reference}}{|score_{reference}|}
\]

This permits one common axis across impact categories with different physical units. Zero reference scores are exported with an undefined relative change and omitted from the plot rather than replaced silently.


In [ ]:
def scenario_comparison_table(raw_scores, market, reference_year):
    selected = raw_scores.loc[raw_scores["market"].eq(market)].copy()
    reference = (
        selected.loc[selected["year"].eq(reference_year),
                     ["scenario key", "method", "score per kg H2"]]
        .rename(columns={"score per kg H2": "reference score per kg H2"})
    )
    expected_scenarios = set(selected["scenario key"])
    missing_reference = sorted(
        expected_scenarios - set(reference["scenario key"])
    )
    if missing_reference:
        raise ValueError(
            f"Market {market!r} has no {reference_year} reference for scenarios: "
            f"{missing_reference}"
        )
    compared = selected.merge(
        reference,
        on=["scenario key", "method"],
        validate="many_to_one",
    )
    compared["change from reference (%)"] = np.where(
        compared["reference score per kg H2"] != 0,
        100.0
        * (compared["score per kg H2"] - compared["reference score per kg H2"])
        / compared["reference score per kg H2"].abs(),
        np.nan,
    )
    return compared


def plot_iam_scenario_comparison(comparison, market, reference_year):
    comparison = comparison.loc[comparison["year"].ne(reference_year)].copy()
    if comparison.empty:
        raise ValueError("At least one non-reference year is required for comparison.")

    scenario_order = (
        database_config_df[["scenario key", "scenario label"]]
        .drop_duplicates()["scenario key"]
        .tolist()
    )
    scenario_labels = (
        database_config_df[["scenario key", "scenario label"]]
        .drop_duplicates()
        .set_index("scenario key")["scenario label"]
        .to_dict()
    )
    impact_order = [METHOD_LABELS[method] for method in SELECTED_METHODS]
    years = sorted(comparison["year"].unique())
    missing_year_colors = sorted(set(years) - set(YEAR_COLORS))
    if missing_year_colors:
        raise KeyError(f"Add YEAR_COLORS entries for: {missing_year_colors}")

    finite = comparison["change from reference (%)"].to_numpy(dtype=float)
    finite = finite[np.isfinite(finite)]
    if len(finite):
        xmin = min(0.0, float(finite.min()))
        xmax = max(0.0, float(finite.max()))
        span = max(xmax - xmin, 1.0)
        xlim = (xmin - 0.08 * span, xmax + 0.08 * span)
    else:
        xlim = (-1.0, 1.0)

    fig, axes = plt.subplots(
        1,
        len(scenario_order),
        figsize=(max(8, 5.2 * len(scenario_order) + 4), max(8, 0.55 * len(impact_order) + 2)),
        squeeze=False,
        sharey=True,
    )
    y = np.arange(len(impact_order), dtype=float)
    group_height = 0.78
    bar_height = group_height / len(years)

    for column_index, scenario_key in enumerate(scenario_order):
        ax = axes[0, column_index]
        panel = comparison.loc[comparison["scenario key"].eq(scenario_key)]
        panel_values = panel["change from reference (%)"].to_numpy(dtype=float)
        if not np.isfinite(panel_values).any():
            ax.text(
                0.5, 0.5, f"NA — {market} market unavailable in {reference_year}",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=12, color="#555555", wrap=True,
            )
        for year_index, year in enumerate(years):
            values = (
                panel.loc[panel["year"].eq(year)]
                .set_index("impact label")["change from reference (%)"]
                .reindex(impact_order)
                .to_numpy(dtype=float)
            )
            offset = (year_index - (len(years) - 1) / 2) * bar_height
            ax.barh(
                y + offset,
                values,
                height=bar_height * 0.88,
                color=YEAR_COLORS[year],
                edgecolor="white",
                linewidth=0.4,
                label=str(year),
                zorder=3,
            )
        ax.axvline(0, color="black", linewidth=0.9, zorder=4)
        ax.set_xlim(*xlim)
        ax.set_yticks(y, impact_order)
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.25, linestyle="--", zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_title(scenario_labels[scenario_key], pad=10)
        ax.xaxis.set_label_position("top")
        ax.xaxis.tick_top()
        ax.set_xlabel(f"Change relative to {reference_year} (%)")
        if column_index > 0:
            ax.tick_params(axis="y", labelleft=False)

    handles = [
        mpatches.Patch(facecolor=YEAR_COLORS[year], edgecolor="white", label=str(year))
        for year in years
    ]
    axes[0, -1].legend(handles=handles, title="Year", loc="lower right", frameon=False)
    fig.suptitle(f"IAM scenario comparison — {market} hydrogen market", y=1.01)
    finish_figure(
        fig,
        f"02_iam_scenario_comparison_{safe_filename(market)}.png",
        rect=(0.03, 0.02, 0.99, 0.98),
    )


scenario_comparisons = {}
for market in SCENARIO_COMPARISON_MARKETS:
    if market not in raw_scores_df["market"].unique():
        warnings.warn(f"Skipping unavailable scenario-comparison market: {market}")
        continue
    table = scenario_comparison_table(
        raw_scores_df,
        market=market,
        reference_year=COMPARISON_REFERENCE_YEAR,
    )
    scenario_comparisons[market] = table
    if EXPORT_RESULTS:
        path = OUTPUT_DIR / f"08_scenario_comparison_{safe_filename(market)}.csv"
        table.to_csv(path, index=False)
        exported_paths[f"scenario comparison — {market}"] = path
    plot_iam_scenario_comparison(table, market, COMPARISON_REFERENCE_YEAR)


## 8. Output handoff and interpretation checks

Review the market-selection and classification-audit CSVs before interpreting changes. In particular:

- compare only identical products, units, functional units, and impact methods;
- distinguish generic `RER` from sector-specific IAM-region markets;
- treat absent sector markets as scenario outcomes unless the selection audit indicates a defect;
- preserve negative contributions and shares above 100%, which may represent credits and offsetting burdens;
- do not attribute a change to transport alone until production-mix and distribution components have both been inspected;
- do not interpret any contribution result unless its reconciliation table passes.


In [ ]:
handoff_df = pd.DataFrame([
    {"artifact": label, "path": str(path)}
    for label, path in exported_paths.items()
]).sort_values("artifact")
display(handoff_df)
display(
    raw_scores_df.sort_values(
        ["scenario key", "year", "impact label", "market"]
    )
)
